# 1.2 — Word Count in PySpark

**Chapter 1, section 1.11** (*A First Distributed Program*).

**The question this notebook answers:** what do the six lines of the chapter's word count
actually do, one step at a time?

By this point in the chapter, word count has been seen three times as a concept: as a figure
splitting map from shuffle from reduce, as three separate phase diagrams, and as a
cluster-level picture with the shuffle band in the middle. Here it becomes something you can
run. The point being made is that **the programmer never writes the shuffle** —
`flatMap` and `map` are the map phase, and `reduceByKey` performs the shuffle *and* the
reduce, without either appearing in the code.

The last section supports **Exercise 8**: the same pipeline reading a file from a path, so
that pointing it at object storage and submitting it as a cloud job is a one-line change
rather than a rewrite. The script for that submission is
`code/cloud/01.04 Word Count Cloud Job.py`.

Runs on a laptop in well under a minute.

## Preflight

Before anything else, confirm which Python, which Java, and which Spark this notebook is
about to use. When a PySpark notebook misbehaves on a new machine, the answer is in this cell
about half the time.

In [1]:
import os, sys
import pyspark

def short(path, keep=3):
    """Last few components of a path: enough to tell you which environment you are in,
    without writing somebody's home directory into a committed notebook."""
    if not path:
        return path
    parts = os.path.normpath(path).split(os.sep)
    return "..." + os.sep + os.path.join(*parts[-keep:]) if len(parts) > keep else path

print("python     :", sys.version.split()[0], short(sys.executable))
print("SPARK_HOME :", short(os.environ.get("SPARK_HOME")))
print("JAVA_HOME  :", short(os.environ.get("JAVA_HOME")))
print("pyspark    :", pyspark.__version__)

python     : 3.12.14 .../.venv/bin/python3
SPARK_HOME : None
JAVA_HOME  : .../temurin-21.jdk/Contents/Home
pyspark    : 4.2.0


## Session setup

One `SparkSession`, created once. The `SparkContext` is reached *through* the session rather
than created beside it — the session creates and owns the context — and it is the context
that the RDD API is reached through, which is why this chapter's code uses `sc`.

There is no `sqlContext` and no `findspark` here: both belong to Spark 1.x, and `sqlContext`
does not exist in Spark 4 at all.

In [2]:
# --- CS-777 session setup ------------------------------------------------
# Chapter 1, section 1.11.
import os, tempfile
from pyspark.sql import SparkSession

DATA = os.environ.get("CS777_DATA", "../data")          # -> code/data/
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-1.2")
         .master("local[*]")                            # [*] = one thread per core
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

sc = spark.sparkContext        # the RDD API lives here
print("Spark  :", spark.version)
print("master :", sc.master)

Spark  : 4.2.0
master : local[*]


## 1. The corpus, and the map phase

`parallelize` takes a Python list that is already in the driver's memory and distributes it
across the executors. It is for examples and tests: in production the data is far too large to
sit in the driver, and `textFile` is used instead — as it is in section 4 below.

`flatMap` applies a function that returns a *sequence* and flattens the result: one line in,
many words out. This is the operation that turns lines into words, and it is the reason
`flatMap` rather than `map` is the first step of every word count ever written.

In [3]:
lines = sc.parallelize([
    "Apache Spark is a unified analytics engine for large-scale data processing.",
    "It provides high-level APIs in Java, Scala, Python and R",
    "it also supports a rich set of higher-level tools including Spark SQL",
    "MLlib for machine learning",
    "GraphX for graph processing",
    "Structured Streaming for incremental computation and stream processing",
])

words = lines.flatMap(lambda x: x.split(' '))

print("lines :", lines.count(), "records")
print("words :", words.count(), "records   <- flatMap turned 6 records into many\n")
print(words.collect())

lines : 6 records
words : 49 records   <- flatMap turned 6 records into many

['Apache', 'Spark', 'is', 'a', 'unified', 'analytics', 'engine', 'for', 'large-scale', 'data', 'processing.', 'It', 'provides', 'high-level', 'APIs', 'in', 'Java,', 'Scala,', 'Python', 'and', 'R', 'it', 'also', 'supports', 'a', 'rich', 'set', 'of', 'higher-level', 'tools', 'including', 'Spark', 'SQL', 'MLlib', 'for', 'machine', 'learning', 'GraphX', 'for', 'graph', 'processing', 'Structured', 'Streaming', 'for', 'incremental', 'computation', 'and', 'stream', 'processing']


In [4]:
# Still the map phase: one record in, one record out.  Every word becomes (word, 1).
rdd1 = words.map(lambda x: (x, 1))
print(rdd1.take(12))

[('Apache', 1), ('Spark', 1), ('is', 1), ('a', 1), ('unified', 1), ('analytics', 1), ('engine', 1), ('for', 1), ('large-scale', 1), ('data', 1), ('processing.', 1), ('It', 1)]


## 2. The shuffle, and the reduce

`reduceByKey` is the whole right-hand side of the diagram. It moves every pair sharing a word
to a single location — that is the shuffle — and sums the values there. Neither the movement
nor the grouping appears anywhere in the code below; the programmer supplies only the two
functions on either side of the shuffle.

In [5]:
rdd2 = rdd1.reduceByKey(lambda x, y: x + y)
print(rdd2.collect())

[('and', 2), ('is', 1), ('APIs', 1), ('Structured', 1), ('Scala,', 1), ('of', 1), ('graph', 1), ('set', 1), ('higher-level', 1), ('engine', 1), ('high-level', 1), ('tools', 1), ('for', 4), ('large-scale', 1), ('provides', 1), ('Java,', 1), ('including', 1), ('It', 1), ('Python', 1), ('it', 1), ('learning', 1), ('Spark', 2), ('in', 1), ('processing', 2), ('supports', 1), ('SQL', 1), ('machine', 1), ('unified', 1), ('data', 1), ('rich', 1), ('Streaming', 1), ('incremental', 1), ('analytics', 1), ('MLlib', 1), ('GraphX', 1), ('a', 2), ('Apache', 1), ('processing.', 1), ('computation', 1), ('R', 1), ('also', 1), ('stream', 1)]


In [6]:
# top(n, key) is an action: it brings a small, already-reduced result back to the driver.
top = rdd2.top(10, lambda x: x[1])
for word, count in top:
    print(f"{count:3d}  {word}")

  4  for
  2  and
  2  Spark
  2  processing
  2  a
  1  is
  1  APIs
  1  Structured
  1  Scala,
  1  of


## 3. Normalizing first

The counts above are wrong, in a way that matters more the larger the corpus gets. Look for
`processing` and `processing.` in the list: they are counted as two different words, because
one of them has a full stop stuck to it. `It` and `it` are counted separately too, and so are
`Java,` and `Java` would be if the corpus contained both.

The normalized pipeline lower-cases each word and strips the punctuation before counting. It
is the same computation with two extra `map` steps — both narrow, both free, both applied
inside the map phase before anything crosses the network.

In [7]:
# The chapter's version, as one chain.
top_words = (lines
    .flatMap(lambda x: x.split(' '))
    .map(lambda x: x.lower())
    .map(lambda x: x.replace(".", "").replace(",", ""))
    .map(lambda x: (x, 1))
    .reduceByKey(lambda x, y: x + y)
    .top(10, lambda x: x[1]))

for word, count in top_words:
    print(f"{count:3d}  {word}")

  4  for
  3  processing
  2  and
  2  it
  2  spark
  2  a
  1  java
  1  is
  1  of
  1  graph


In [8]:
# The same thing written as three named RDDs, which is easier to inspect while debugging.
# Nothing here computes anything: both cells are transformations only.
words1 = (lines
          .flatMap(lambda x: x.split(' '))
          .map(lambda x: x.lower())
          .map(lambda x: x.replace(".", "").replace(",", "")))

words2 = words1.map(lambda x: (x, 1)).reduceByKey(lambda x, y: x + y)

# ...and only now does anything run.
print(words2.top(10, lambda x: x[1]))

[('for', 4), ('processing', 3), ('and', 2), ('it', 2), ('spark', 2), ('a', 2), ('java', 1), ('is', 1), ('of', 1), ('graph', 1)]


In [9]:
# The two forms must agree.  An assertion that runs is worth more than a sentence saying so.
assert sorted(top_words) == sorted(words2.top(10, lambda x: x[1]))
print("chained form and named-RDD form agree")

# And normalization really did merge words that were counted separately before.
raw = dict(rdd2.collect())
split_pairs = [("processing", "processing."), ("It", "it")]
print("\nCounted separately before normalizing:")
for a, b in split_pairs:
    print(f"  {a!r:16s} {raw.get(a, 0)}   +   {b!r:16s} {raw.get(b, 0)}")
merged = dict(words2.filter(lambda kv: kv[0] in ("processing", "it")).collect())
print("\nCounted together after normalizing:")
for w in ("processing", "it"):
    print(f"  {w!r:16s} {merged[w]}")

chained form and named-RDD form agree



Counted separately before normalizing:
  'processing'     2   +   'processing.'    1
  'It'             1   +   'it'             1

Counted together after normalizing:
  'processing'     3
  'it'             2


## 4. The same pipeline, reading a file

*This section is the starting point for **Exercise 8**.*

Everything above built its corpus with `parallelize`, which requires the data to fit in the
driver. Real input is read from storage with `textFile`, and the only thing that changes is
the first line: `textFile` accepts a local path, an `hdfs://` path, or an object-storage URI
such as `gs://` or `s3a://`, and the executors read it directly from wherever it is.

To run Exercise 8, change `CORPUS` to a `gs://` URI pointing at a file of at least several
hundred megabytes, then submit `code/cloud/01.04 Word Count Cloud Job.py` as a serverless job
following the procedure in the cloud appendix. The pipeline below and the pipeline in that
script are deliberately identical, so that the only difference between your local run and your
cloud run is the size and the location of the input.

In [10]:
CORPUS = os.path.join(DATA, "Alices-Adventures-in-Wonderland-by-Lewis-Carroll.txt.bz2")

# For Exercise 8, replace the line above with an object-storage URI, for example:
# CORPUS = "gs://your-bucket/some-large-corpus.txt"
#
# bzip2 is used here deliberately: it is a *splittable* compression format, so Spark can
# divide the file across tasks.  A .gz file cannot be split, and however large it is, it
# will be read by exactly one task.

print("reading:", CORPUS)
file_lines = sc.textFile(CORPUS)
print("lines  :", file_lines.count())

reading: ../data/Alices-Adventures-in-Wonderland-by-Lewis-Carroll.txt.bz2


lines  : 3774


In [11]:
import re, time

# The identical normalized pipeline, applied to the file instead of to the list.
# re.findall is used instead of the two .replace() calls: on real prose there is more
# punctuation than a full stop and a comma.  The apostrophe class carries both the ASCII
# form and U+2019, which is what this Gutenberg text actually uses -- without it, "don't"
# splits into "don" and "t".
t0 = time.time()
file_counts = (file_lines
    .flatMap(lambda line: re.findall(r"[a-z'’]+", line.lower()))
    .map(lambda w: (w, 1))
    .reduceByKey(lambda a, b: a + b))

top20 = file_counts.top(20, key=lambda kv: kv[1])
elapsed = time.time() - t0

print(f"distinct words : {file_counts.count():,}")
print(f"ELAPSED_SECONDS={elapsed:.2f}")
print("     ^ machine-dependent.  This is the number Exercise 8 asks you to compare against")
print("       the same pipeline submitted as a cloud job.  Expect the cloud run to be the")
print("       SLOWER of the two on a small file: you will be timing job submission, machine")
print("       allocation and JVM startup, not the word count.\n")
for word, count in top20:
    print(f"{count:5d}  {word}")

distinct words : 3,102
ELAPSED_SECONDS=0.05
     ^ machine-dependent.  This is the number Exercise 8 asks you to compare against
       the same pipeline submitted as a cloud job.  Expect the cloud run to be the
       SLOWER of the two on a small file: you will be timing job submission, machine
       allocation and JVM startup, not the word count.

 1837  the
  946  and
  811  to
  695  a
  637  of
  543  it
  541  she
  462  said
  439  you
  437  in
  411  i
  386  alice
  359  was
  295  that
  273  as
  248  her
  229  with
  225  at
  204  on
  200  all


## Conclusion

The six lines of section 1.11 map exactly onto the three phases of the MapReduce model
developed in section 1.9:

| Phase | In the code |
|-------|-------------|
| **Map** | `flatMap(split)` and `map(lambda w: (w, 1))` — each record handled independently |
| **Shuffle** | inside `reduceByKey` — never written by the programmer |
| **Reduce** | the `lambda a, b: a + b` given to `reduceByKey` |

Two things are worth carrying forward. First, the combining function passed to `reduceByKey`
must be **associative**, because the system, not the programmer, chooses the grouping;
addition is, which is why word count is the canonical example. Second, changing the input from
a Python list to a file in object storage changed exactly one line — which is what makes
Exercise 8 a matter of scale rather than of rewriting.

**Next.** Notebook 1.3 places sequential Python and PySpark side by side for six algorithms,
and asks what the difference between them really is.